# Exkurs: Ein eigenes Mini-LLM bauen

> 🧭 **Dies ist ein optionaler Exkurs.** Er gehört nicht zum Pflichtstoff, sondern ist eine
> Einladung an alle, die einmal *wirklich* verstehen wollen, was hinter einem Large Language
> Model (LLM) steckt — jenseits von Buzzwords wie "künstliche Intelligenz" oder "generative KI".
> Wenn Du diesen Exkurs überspringst, verpasst Du keinen Prüfungsstoff. Wenn Du ihn machst,
> nimmst Du eine der wertvollsten Erkenntnisse dieses gesamten Kurses mit.

## Real-World-Kontext

In der Vorlesung hast Du das Projekt **"microgpt"** von Andrej Karpathy kennengelernt: ein
funktionsfähiges Sprachmodell in nur ~200 Zeilen Python mit gerade einmal 4.192 Parametern,
trainiert auf 32.000 englischen Namen. Die Vorlesungsfolie hat Dir die **Konzepte** gezeigt —
Character Embedding, Positional Encoding, ein Transformer-Block, Autoregression,
Temperature-Sampling — aber bewusst **keinen vollständigen Code**. Das holen wir jetzt nach.

Wir bleiben nah am Original-Gedanken, bauen unsere eigene, für diesen Kurs vereinfachte Version
aber komplett selbst mit **PyTorch** (statt Karpathys handgeschriebener Autograd-Engine — das
Grundprinzip bleibt identisch). Als Datensatz nutzen wir keine englischen, sondern **deutsche
Vornamen**. Der motivierende Rahmen dafür: Unser Kfz-Versicherer aus dem Running Example braucht
gelegentlich **synthetische Testdaten** — z. B. plausibel klingende, aber komplett erfundene
Kundennamen für eine Testumgebung, ohne echte Kundendaten zu verwenden. Ein winziges,
selbstgebautes Sprachmodell, das "deutsche Vornamen" gelernt hat, kann genau das leisten.

> 💡 **Good to know:**
> Alles, was Du in diesem Notebook baust, ist **eigenständig implementierter Code** — keine
> Kopie von Karpathys microgpt. Wir übernehmen die *Idee* (winziges Transformer-Modell,
> Character-Level, Namen als Trainingsdaten), nutzen dafür aber PyTorch-Bausteine, die Du schon
> aus den letzten Kapiteln kennst, statt eine eigene Autograd-Engine nachzubauen.

## 🧭 So bearbeitest Du dieses Notebook

Führe die Zellen **von oben nach unten** aus (`Shift + Enter`). Jede Code-Zelle beginnt mit einer Markierung, die zeigt, wie viel Du selbst tust:

| Markierung | Was Du tust |
|---|---|
| **I DO** | Ich zeige es vor. Zelle ausführen, Ausgabe lesen, Kommentare nachvollziehen. |
| **WE DO** | Wir machen es gemeinsam. Ausführen, die Ausgabe deuten und bei Bedarf kleine Werte selbst ändern. |
| **YOU DO / 🎯 EXERCISE** | Du bist dran. Ersetze die Platzhalter (`???` bzw. `TODO`) durch eigenen Code oder Deine eigene Antwort und führe die Zelle aus. |

Ein Kasten **🎯 Your Task** vor einer Zelle beschreibt, was genau zu tun ist. Kommt eine Fehlermeldung, hast Du vermutlich eine Zelle weiter oben noch nicht ausgeführt.

## 🎯 Learning Objectives

By completing this notebook, you will be able to:
- 💡 **Explain**, warum ein LLM algorithmisch kein "Buzzword-Monster" ist, sondern aus denselben
  Bausteinen besteht wie die neuronalen Netze aus den vorigen Kapiteln (Embeddings, lineare
  Schichten, Aktivierungsfunktionen, Backpropagation).
- 🔍 **Identify** die vier Kernkomponenten eines Transformer-basierten Sprachmodells (Character
  Embedding, Positional Encoding, Self-Attention + MLP im Transformer-Block, Output-Projektion)
  im tatsächlichen PyTorch-Code wieder.
- 🛠️ **Apply** einen kompletten Trainingsloop (Forward Pass → Loss → `zero_grad()` →
  `backward()` → `step()`) auf ein eigenes, kleines Sprachmodell.
- 🛠️ **Apply** Temperature-Sampling, um bei der Text-Generierung gezielt zwischen
  deterministischem, balanciertem und zufälligem Verhalten zu steuern.
- ⚖️ **Bewerten**, welche Stellschrauben (Datenmenge, Parameteranzahl, Trainingsschritte) ein
  winziges Modell wie dieses von einem Produktions-LLM wie GPT-4 unterscheiden.

## Concept at a Glance

**Analogie — das Sprachmodell als Rate-Automat mit Gedächtnis:** Stell Dir ein Sprachmodell wie
einen sehr aufmerksamen Rate-Automaten vor, der Buchstabe für Buchstabe vorhersagt: "Wenn bisher
`A-N-N` dastand — was kommt wahrscheinlich als Nächstes?" Nach sehr vielem Training (also nach
sehr vielem "Buchstaben-Raten und aus Fehlern lernen") hat der Automat gelernt, dass nach `ANN`
oft ein `A` kommt (→ "ANNA"), manchmal ein `E` (→ "ANNE") — aber praktisch nie ein `X`. Ein LLM
wie ChatGPT macht exakt dasselbe, nur nicht mit Buchstaben von Namen, sondern mit
**Wort-Bruchstücken (Tokens)** von praktisch dem gesamten Internet-Text — und mit Milliarden
statt tausenden Parametern.

**Die vier Bausteine, die wir gleich bauen** (identisch zur Vorlesungsfolie
"Architektur-Übersicht (microgpt)"):

1. **Character Embedding** — jedes Zeichen wird in einen Vektor aus Zahlen übersetzt (das Netz
   kann nicht direkt mit Buchstaben rechnen, nur mit Zahlen).
2. **Positional Encoding** — das Modell erfährt zusätzlich, *an welcher Stelle* im Namen ein
   Zeichen steht (ohne das wüsste es nicht, ob "N" das erste oder das letzte Zeichen ist).
3. **Transformer-Block** — Self-Attention (jedes Zeichen "schaut" auf die vorigen Zeichen) plus
   ein kleines Feed-Forward-Netz (MLP), beides mit Residual Connections und Normalisierung.
4. **Output-Projektion** — am Ende wird der finale Vektor zurück in eine Wahrscheinlichkeit pro
   möglichem nächsten Zeichen übersetzt.

> 💡 **Good to know:**
> Wir verwenden hier **LayerNorm** statt der auf der Vorlesungsfolie erwähnten **RMSNorm** — beide
> normalisieren Zwischenwerte, damit das Training stabil bleibt, RMSNorm ist nur eine
> recheneffizientere Variante. Für unser Verständnis macht das keinen Unterschied.

## Schritt 1 — Die Datengrundlage: Eine Liste deutscher Vornamen

Wir brauchen keinen Download und keinen externen Datensatz. Wir bauen unsere Trainingsdaten
einfach als **Python-Liste** aus ca. 340 gängigen deutschen Vor- und Rufnamen — männlich und
weiblich gemischt.

In [ ]:
# I DO: Liste deutscher Vornamen als Trainingsdatensatz definieren

alle_namen = [
    "Alexander",
    "Andreas",
    "Anton",
    "Arne",
    "Bastian",
    "Benedikt",
    "Benjamin",
    "Bernd",
    "Bjoern",
    "Christian",
    "Christoph",
    "Claas",
    "Daniel",
    "David",
    "Dennis",
    "Dieter",
    "Dominik",
    "Elias",
    "Emil",
    "Erik",
    "Fabian",
    "Felix",
    "Ferdinand",
    "Finn",
    "Florian",
    "Frank",
    "Franz",
    "Friedrich",
    "Georg",
    "Gerhard",
    "Gunnar",
    "Hannes",
    "Hans",
    "Heinrich",
    "Heinz",
    "Helmut",
    "Henning",
    "Henry",
    "Hermann",
    "Holger",
    "Ingo",
    "Jakob",
    "Jan",
    "Janosch",
    "Jens",
    "Joachim",
    "Johann",
    "Johannes",
    "Jonas",
    "Jonathan",
    "Josef",
    "Julian",
    "Juergen",
    "Justus",
    "Karl",
    "Karsten",
    "Kevin",
    "Klaus",
    "Konstantin",
    "Kurt",
    "Lars",
    "Leon",
    "Leonard",
    "Leopold",
    "Liam",
    "Lorenz",
    "Luca",
    "Lukas",
    "Malte",
    "Manfred",
    "Manuel",
    "Marc",
    "Marcel",
    "Marco",
    "Mario",
    "Mark",
    "Markus",
    "Marlon",
    "Martin",
    "Marius",
    "Mats",
    "Matthias",
    "Matteo",
    "Max",
    "Maximilian",
    "Michael",
    "Milan",
    "Moritz",
    "Nick",
    "Niklas",
    "Nils",
    "Noah",
    "Norbert",
    "Oliver",
    "Oskar",
    "Otto",
    "Patrick",
    "Paul",
    "Peter",
    "Philipp",
    "Rainer",
    "Ralf",
    "Raphael",
    "Reiner",
    "Rene",
    "Richard",
    "Robert",
    "Robin",
    "Roland",
    "Rolf",
    "Rudolf",
    "Ruediger",
    "Sascha",
    "Sebastian",
    "Sepp",
    "Simon",
    "Stefan",
    "Stephan",
    "Sven",
    "Theo",
    "Thomas",
    "Thorsten",
    "Tim",
    "Timo",
    "Tobias",
    "Torben",
    "Udo",
    "Ulrich",
    "Uwe",
    "Vincent",
    "Volker",
    "Walter",
    "Werner",
    "Wilhelm",
    "Willi",
    "Wolfgang",
    "Yannick",
    "Hans-Peter",
    "Karl-Heinz",
    "Klaus-Dieter",
    "Jan-Niklas",
    "Paul-Erik",
    "Adele",
    "Agnes",
    "Alexandra",
    "Alina",
    "Alma",
    "Amelie",
    "Andrea",
    "Angelika",
    "Anja",
    "Anke",
    "Anna",
    "Annalena",
    "Anneliese",
    "Annette",
    "Antje",
    "Astrid",
    "Barbara",
    "Beate",
    "Bettina",
    "Birgit",
    "Brigitte",
    "Carla",
    "Carmen",
    "Carola",
    "Caroline",
    "Cathrin",
    "Charlotte",
    "Christel",
    "Christiane",
    "Christina",
    "Christine",
    "Clara",
    "Claudia",
    "Constanze",
    "Cornelia",
    "Dagmar",
    "Daniela",
    "Denise",
    "Diana",
    "Doreen",
    "Doris",
    "Dorothea",
    "Edith",
    "Elena",
    "Elfriede",
    "Elisabeth",
    "Elke",
    "Ella",
    "Else",
    "Emilia",
    "Emily",
    "Emma",
    "Erika",
    "Erna",
    "Eva",
    "Finja",
    "Franziska",
    "Frauke",
    "Frieda",
    "Frida",
    "Gabriele",
    "Gerda",
    "Gertrud",
    "Gisela",
    "Greta",
    "Gudrun",
    "Hanna",
    "Hannah",
    "Hannelore",
    "Heide",
    "Heidi",
    "Heike",
    "Helene",
    "Helga",
    "Henriette",
    "Hilde",
    "Ida",
    "Ilse",
    "Ines",
    "Ingeborg",
    "Inge",
    "Ingrid",
    "Irene",
    "Iris",
    "Isabel",
    "Isolde",
    "Jana",
    "Jasmin",
    "Jennifer",
    "Johanna",
    "Josefine",
    "Judith",
    "Julia",
    "Juliane",
    "Jutta",
    "Karin",
    "Karla",
    "Katharina",
    "Katja",
    "Katrin",
    "Kerstin",
    "Klara",
    "Kristin",
    "Laura",
    "Lea",
    "Leni",
    "Leonie",
    "Lena",
    "Lian",
    "Lina",
    "Lisa",
    "Lotte",
    "Louisa",
    "Luisa",
    "Luise",
    "Lydia",
    "Magdalena",
    "Manuela",
    "Margarete",
    "Margot",
    "Maria",
    "Marie",
    "Marion",
    "Marlene",
    "Marta",
    "Martha",
    "Martina",
    "Mathilde",
    "Mathilda",
    "Meike",
    "Melanie",
    "Merle",
    "Michaela",
    "Mila",
    "Milena",
    "Mira",
    "Monika",
    "Nadine",
    "Nadja",
    "Nele",
    "Nicole",
    "Nina",
    "Nora",
    "Olga",
    "Paula",
    "Paulina",
    "Petra",
    "Pia",
    "Ramona",
    "Regina",
    "Renate",
    "Rita",
    "Romina",
    "Romy",
    "Ronja",
    "Rosa",
    "Rosemarie",
    "Ruth",
    "Sabine",
    "Sabrina",
    "Sandra",
    "Sara",
    "Selina",
    "Selma",
    "Silke",
    "Silvia",
    "Simone",
    "Sofie",
    "Sonja",
    "Sophia",
    "Stefanie",
    "Stella",
    "Susanne",
    "Svenja",
    "Sylvia",
    "Tanja",
    "Thea",
    "Theresa",
    "Ulrike",
    "Ursula",
    "Ute",
    "Valentina",
    "Valeria",
    "Vanessa",
    "Vera",
    "Veronika",
    "Viktoria",
    "Wanda",
    "Waltraud",
    "Wiebke",
    "Xenia",
    "Yara",
    "Yvonne",
    "Zoe",
    "Zora",
    "Anna-Lena",
    "Eva-Maria",
    "Marie-Luise",
    "Ann-Kathrin",
    "Maria-Theresia"
]

print(f"Anzahl Namen im Datensatz: {len(alle_namen)}")
print(f"Beispiele: {alle_namen[:5]} ... {alle_namen[-5:]}")
print(f"Kuerzester Name: {min(alle_namen, key=len)} ({len(min(alle_namen, key=len))} Zeichen)")
print(f"Laengster Name:  {max(alle_namen, key=len)} ({len(max(alle_namen, key=len))} Zeichen)")

### Zwischenfazit

Wir haben unsere Trainingsdaten — eine schlichte Liste aus Text-Strings. Ein neuronales Netz
kann mit Buchstaben aber nichts anfangen, es rechnet ausschließlich mit Zahlen. Der nächste
Schritt: ein **Character-Level-Tokenizer**, der jeden vorkommenden Buchstaben in eine eindeutige
Zahl übersetzt.

## Schritt 2 — Character-Level-Tokenizer bauen

**Analogie — der Tokenizer als Telefonbuch:** Ein Tokenizer ist wie ein Telefonbuch, das jedem
Zeichen eine eindeutige Nummer zuweist: "a" → 1, "b" → 2, "c" → 3, und so weiter. Zusätzlich
brauchen wir drei **Spezial-Tokens**:

- `<BOS>` (*Beginning of Sequence*) — markiert den Namensanfang.
- `<EOS>` (*End of Sequence*) — markiert das Namensende (das Modell "weiß" dadurch, wann es
  aufhören soll, weitere Zeichen zu generieren).
- `<PAD>` (*Padding*) — ein Lückenfüller, damit alle Namen im Datensatz auf dieselbe Länge
  gebracht werden können (Namen sind unterschiedlich lang, ein Netz braucht aber feste
  Eingabegrößen pro Batch).

> ⚠️ **Common Pitfall:**
> `<BOS>`, `<EOS>` und `<PAD>` sind keine Zauberei — es sind einfach drei zusätzliche "Buchstaben"
> in unserem Vokabular, denen wir ebenfalls eine Zahl zuweisen. Das Modell behandelt sie beim
> Rechnen genau wie jeden anderen Buchstaben auch.

In [ ]:
# I DO: Vokabular aus allen vorkommenden Zeichen aufbauen

# Alle einzigartigen Zeichen aus allen Namen sammeln (set() entfernt Duplikate)
alle_zeichen = sorted(set("".join(alle_namen)))

# Spezial-Tokens ergaenzen: Namensanfang, Namensende, Auffuell-Zeichen
spezial_tokens = ["<PAD>", "<BOS>", "<EOS>"]
vokabular = spezial_tokens + alle_zeichen

# Zwei Dictionaries: Zeichen -> Zahl und Zahl -> Zeichen (fuer die Rueckuebersetzung)
char_to_idx = {char: idx for idx, char in enumerate(vokabular)}
idx_to_char = {idx: char for idx, char in enumerate(vokabular)}

vokabular_groesse = len(vokabular)
print(f"Vokabular ({vokabular_groesse} Zeichen/Tokens gesamt):")
print(vokabular)

Testen wir den Tokenizer an einem Beispielnamen: Wir kodieren ihn zu Zahlen und übersetzen
ihn direkt wieder zurück — wenn beides übereinstimmt, funktioniert unser Telefonbuch korrekt.

In [ ]:
# I DO: Einen Beispielnamen kodieren und wieder dekodieren

beispiel_name = "Anna"

# Kodieren: <BOS> + jedes Zeichen + <EOS>, jeweils als Zahl
kodiert = [char_to_idx["<BOS>"]] + [char_to_idx[c] for c in beispiel_name] + [char_to_idx["<EOS>"]]
print(f"'{beispiel_name}' kodiert:", kodiert)

# Dekodieren: jede Zahl zurueck in ein Zeichen uebersetzen
dekodiert = "".join(idx_to_char[i] for i in kodiert)
print(f"Zurueck dekodiert: {dekodiert}")

### Zwischenfazit

Der Tokenizer funktioniert. Jetzt müssen wir alle 342 Namen einheitlich lang machen (Padding),
damit wir sie gemeinsam als einen einzigen Zahlen-Tensor an PyTorch übergeben können.

## Schritt 3 — Alle Namen auf feste Länge bringen

Wir wählen eine feste **maximale Sequenzlänge** (`max_laenge`) — lang genug für den längsten
Namen plus `<BOS>` und `<EOS>`. Kürzere Namen werden mit `<PAD>` aufgefüllt.

In [ ]:
# I DO: Feste Sequenzlaenge bestimmen und Kodier-Funktion definieren

max_laenge = max(len(n) for n in alle_namen) + 2  # +2 fuer <BOS> und <EOS>
print(f"Maximale Sequenzlaenge: {max_laenge}")

def kodiere_namen(name, max_laenge):
    """Wandelt einen Namen in eine Zahlen-Sequenz fester Laenge um (mit BOS/EOS/PAD)."""
    sequenz = [char_to_idx["<BOS>"]] + [char_to_idx[c] for c in name] + [char_to_idx["<EOS>"]]
    # Mit <PAD> auffuellen, bis max_laenge erreicht ist
    sequenz = sequenz + [char_to_idx["<PAD>"]] * (max_laenge - len(sequenz))
    return sequenz

# Test an einem kurzen Namen
beispiel = kodiere_namen("Emma", max_laenge)
print(f"'Emma' auf Laenge {max_laenge} aufgefuellt: {beispiel}")

In [ ]:
# I DO: Alle Namen kodieren und in einen einzigen PyTorch-Tensor packen

import torch

alle_sequenzen = torch.tensor([kodiere_namen(name, max_laenge) for name in alle_namen])

print(f"Form des Datensatz-Tensors: {tuple(alle_sequenzen.shape)}  (Anzahl Namen, Sequenzlaenge)")
print(f"Erste Zeile (kodiertes '{alle_namen[0]}'):\n{alle_sequenzen[0]}")

### Zwischenfazit

Wir haben jetzt einen einzigen Zahlen-Tensor, der unseren kompletten Trainingsdatensatz enthält.
Zeit für den eigentlichen Kern dieses Exkurses: das Modell selbst.

## Schritt 4 — `MiniGPT`: Die Transformer-Architektur in PyTorch

**Analogie — das Fließband mit drei Stationen:** Stell Dir das Modell als kleines Fließband vor.
Ein kodierter Name läuft durch drei Stationen:

1. **Embedding-Station:** Jede Zahl (Zeichen) wird zu einem Vektor aus lernbaren Zahlen — plus
   ein zweiter Vektor, der nur die *Position* im Namen kodiert. Beide werden addiert.
2. **Transformer-Block-Station:** Hier "schaut" jedes Zeichen per **Self-Attention** auf die
   *vorherigen* Zeichen der Sequenz (nie auf zukünftige — sonst könnte das Modell beim Training
   einfach "abschreiben"!) und sammelt so Kontext ein. Danach verarbeitet ein kleines
   Feed-Forward-Netz (MLP) diesen Kontext weiter. Beide Teilschritte haben eine
   **Residual Connection** (das Original wird addiert, nicht ersetzt) und werden anschließend
   normalisiert.
3. **Output-Projektions-Station:** Der finale Vektor pro Position wird auf `vokabular_groesse`
   Werte projiziert — ein "Wie wahrscheinlich ist jedes mögliche nächste Zeichen?"-Score pro
   Vokabeleintrag.

> 💡 **Good to know:**
> "Self-Attention nur auf vorherige Zeichen" nennt man **causal masking** (kausale Maskierung).
> Ohne diese Einschränkung würde das Modell beim Training in die Zukunft schauen dürfen — beim
> späteren Generieren gibt es aber gar keine "Zukunft" zu sehen, also würde das Modell etwas
> lernen, das es später gar nicht anwenden kann.

In [ ]:
# I DO: MiniGPT als eigene Klasse definieren

import torch.nn as nn

torch.manual_seed(42)  # fester Seed -> reproduzierbare Gewichts-Initialisierung

class MiniGPT(nn.Module):
    def __init__(self, vokabular_groesse, embedding_dim, anzahl_heads, max_laenge):
        super(MiniGPT, self).__init__()
        # 1. Embedding-Station: Zeichen -> Vektor, Position -> Vektor
        self.token_embedding = nn.Embedding(vokabular_groesse, embedding_dim)
        self.position_embedding = nn.Embedding(max_laenge, embedding_dim)

        # 2. Transformer-Block: Self-Attention + MLP, je mit Residual Connection + Norm
        self.attention = nn.MultiheadAttention(embedding_dim, anzahl_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.norm2 = nn.LayerNorm(embedding_dim)

        # 3. Output-Projektion: Vektor -> Score pro Vokabeleintrag
        self.output_projektion = nn.Linear(embedding_dim, vokabular_groesse)

    def forward(self, x):
        seq_laenge = x.size(1)
        positionen = torch.arange(seq_laenge, device=x.device)

        # Embedding-Station: Zeichen-Vektor + Positions-Vektor addieren
        embeddings = self.token_embedding(x) + self.position_embedding(positionen)

        # Causal Mask: verhindert, dass eine Position auf spaetere Positionen "schaut"
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_laenge)

        # Transformer-Block: Self-Attention mit Residual Connection + Normalisierung
        attention_output, _ = self.attention(embeddings, embeddings, embeddings, attn_mask=causal_mask)
        x = self.norm1(embeddings + attention_output)

        # MLP mit Residual Connection + Normalisierung
        mlp_output = self.mlp(x)
        x = self.norm2(x + mlp_output)

        # Output-Projektion: Vektor pro Position -> Score pro moeglichem naechsten Zeichen
        logits = self.output_projektion(x)
        return logits

# Kleine Architektur, aehnliche Groessenordnung wie microgpt aus der Vorlesungsfolie
embedding_dim = 32
anzahl_heads = 4

modell = MiniGPT(vokabular_groesse, embedding_dim, anzahl_heads, max_laenge)
anzahl_parameter = sum(p.numel() for p in modell.parameters())
print(modell)
print(f"\nAnzahl trainierbarer Parameter: {anzahl_parameter:,}")

> 💡 **Good to know:**
> Unser Modell hat deutlich mehr Parameter als Karpathys 4.192 aus der Vorlesungsfolie — das
> liegt vor allem am größeren Vokabular (deutsche Umlaute, mehr Zeichen) und daran, dass wir mit
> `nn.MultiheadAttention` eine etwas vollständigere Attention-Implementierung nutzen. Die
> Größenordnung (ein paar Zehntausend statt Milliarden Parameter) bleibt aber vergleichbar winzig.

Ein kurzer Sanity-Check: Schicken wir einen echten Batch aus unserem Datensatz einmal durchs
Modell und schauen uns die Form der Ausgabe an.

In [ ]:
# I DO: Sanity-Check - ein Batch durchs Modell schicken

test_batch = alle_sequenzen[:8]  # die ersten 8 kodierten Namen
test_output = modell(test_batch)

print(f"Form der Eingabe:  {tuple(test_batch.shape)}  (Batch, Sequenzlaenge)")
print(f"Form der Ausgabe:  {tuple(test_output.shape)}  (Batch, Sequenzlaenge, Vokabular_Groesse)")
print("\n-> Pro Zeichen-Position bekommen wir einen Score fuer jedes moegliche naechste Zeichen.")

### Zwischenfazit

Die Architektur steht, und die Ausgabeform stimmt. Aber Achtung: Das Modell hat noch **nichts
gelernt** — die Gewichte sind zufällig initialisiert. Jetzt kommt das Training.

## Schritt 5 — Der Trainingsloop

Bevor wir trainieren, ein Referenzwert: Bei **zufälligem Raten** (jedes der `vokabular_groesse`
Zeichen gleich wahrscheinlich) läge der Cross-Entropy-Loss theoretisch bei
$\ln(\text{vokabular\_groesse})$. Ein gut trainiertes Modell sollte deutlich darunter landen —
exakt das Muster, das auf der Vorlesungsfolie für microgpt gezeigt wurde (Loss sinkt von ~3,3 auf
~2,37).

In [ ]:
# I DO: Referenzwert fuer "zufaelliges Raten" berechnen

import math

zufalls_loss = math.log(vokabular_groesse)
print(f"Loss bei zufaelligem Raten (ln({vokabular_groesse})): {zufalls_loss:.3f}")
print("Ziel: Nach dem Training soll der Loss deutlich darunter liegen.")

Für den Trainingsloop brauchen wir noch eine kleine Batch-Funktion, die uns zufällige
Ausschnitte aus unserem Datensatz-Tensor liefert — ganz ohne `DataLoader`-Klasse, das reicht bei
342 Namen völlig aus.

In [ ]:
# I DO: Einfache Batch-Funktion definieren

def hole_batch(batch_groesse):
    """Waehlt zufaellig 'batch_groesse' Namen aus dem Datensatz aus."""
    zufalls_indizes = torch.randint(0, len(alle_sequenzen), (batch_groesse,))
    return alle_sequenzen[zufalls_indizes]

test_batch = hole_batch(4)
print(f"Zufaelliger Test-Batch, Form: {tuple(test_batch.shape)}")

Jetzt der eigentliche Trainingsloop — genau dasselbe **Fünf-Schritte-Muster** wie in den
vorigen Kapiteln (Forward Pass → Loss → `zero_grad()` → `backward()` → `step()`). Eine
Besonderheit: Wir sagen für **jede Position** in der Sequenz das jeweils **nächste** Zeichen
voraus (Eingabe = Sequenz ohne letztes Zeichen, Ziel = Sequenz ohne erstes Zeichen, also um eins
verschoben). `ignore_index` sorgt dafür, dass `<PAD>`-Auffüllzeichen den Loss nicht verfälschen.

> 🎯 **Your Task:**
> Vervollständige die `# TODO`-Zeilen im Trainingsloop unten. Orientiere Dich am Muster aus den
> vorigen Kapiteln.

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Trainingsloop vervollstaendigen
# =========================================================================
# Instruction: Ersetze die TODO-Kommentare durch gueltigen Code (siehe Anleitung oben).

torch.manual_seed(42)
modell = MiniGPT(vokabular_groesse, embedding_dim, anzahl_heads, max_laenge)

pad_idx = char_to_idx["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)  # <PAD> zaehlt nicht zum Loss
optimizer = torch.optim.Adam(modell.parameters(), lr=0.003)

anzahl_schritte = 1500
batch_groesse = 32
loss_verlauf = []

for schritt in range(anzahl_schritte):
    batch = hole_batch(batch_groesse)
    eingabe = batch[:, :-1]    # alle Zeichen ausser dem letzten
    ziel = batch[:, 1:]        # alle Zeichen ausser dem ersten (= "naechstes Zeichen" pro Position)

    # 1. Forward Pass: Vorhersage des Modells fuer diesen Batch berechnen
    logits = ???

    # 2. Loss berechnen: CrossEntropyLoss erwartet die Form (Batch*Sequenz, Vokabular),
    #    deshalb formen wir logits und ziel vorher mit .reshape() um
    loss = criterion(logits.reshape(-1, vokabular_groesse), ziel.reshape(-1))

    # 3. Gradienten aus dem letzten Schritt zuruecksetzen
    ???

    # 4. Backward Pass: Gradienten fuer alle Gewichte berechnen
    ???

    # 5. Gewichte anhand der Gradienten anpassen
    ???

    loss_verlauf.append(loss.item())
    if (schritt + 1) % 250 == 0:
        print(f"Schritt {schritt + 1}/{anzahl_schritte}: Loss = {loss.item():.3f}")

print("Training abgeschlossen.")

> 💡 **Good to know:**
> `logits.reshape(-1, vokabular_groesse)` fasst die Dimensionen "Batch" und "Sequenzposition" zu
> einer einzigen langen Liste von Vorhersagen zusammen (`CrossEntropyLoss` erwartet pro Vorhersage
> **eine** Zeile). `ziel.reshape(-1)` macht dasselbe fuer die Zielwerte. Das ist reine
> "Buchhaltung" der Tensor-Formen, kein neues Konzept.

Werfen wir jetzt einen Blick auf die Loss-Kurve — genau wie in den vorigen Kapiteln.

In [ ]:
# I DO: Loss-Kurve ueber alle Trainingsschritte plotten

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(loss_verlauf, color="tab:purple", linewidth=1)
plt.axhline(y=zufalls_loss, color="tab:red", linestyle="--", label=f"Zufaelliges Raten (ln({vokabular_groesse}) = {zufalls_loss:.2f})")
plt.xlabel("Trainingsschritt")
plt.ylabel("Cross-Entropy-Loss")
plt.title("Trainingsverlauf: MiniGPT lernt deutsche Vornamen")
plt.legend()
plt.show()

print(f"Start-Loss (Schritt 1):  {loss_verlauf[0]:.3f}")
print(f"End-Loss (letzte 50 Schritte, gemittelt): {sum(loss_verlauf[-50:]) / 50:.3f}")

> 💡 **Good to know:**
> Das Muster deckt sich mit der Vorlesungsfolie "Training & Loss-Minimierung": Der Loss fällt zu
> Beginn steil, dann immer flacher — und landet deutlich unter dem Referenzwert für zufälliges
> Raten. Das Modell hat also tatsächlich etwas über die Struktur deutscher Vornamen gelernt
> (z. B. dass nach `<BOS>` selten ein "x" kommt, oder dass auf "sch" oft ein Vokal folgt).

### Zwischenfazit

Unser Mini-Modell trainiert und lernt sichtbar. Bevor wir zur Text-Generierung kommen, probieren
wir aus, wie sich ein verändertes Hyperparameter auf das Endergebnis auswirkt.

## Schritt 6 — Experimentieren: Wie wirken sich Hyperparameter aus?

> 🎯 **Your Task:**
> Verändere in der Zelle unten **einen** der beiden markierten Werte — entweder
> `anzahl_schritte_neu` (z. B. auf `300` senken, um zu sehen, wie ein "zu kurz trainiertes"
> Modell aussieht) oder `embedding_dim_neu` (z. B. auf `8` senken, ein winzigeres Modell) — und
> beobachte den Effekt auf den finalen Loss im Vergleich zum Original-Training oben.

In [ ]:
# WE DO: Hyperparameter veraendern und Effekt auf den finalen Loss beobachten

anzahl_schritte_neu = 300     # <- veraendere diesen Wert, z.B. auf 300 (kuerzeres Training)
embedding_dim_neu = 32        # <- oder veraendere diesen Wert, z.B. auf 8 (kleineres Modell)

torch.manual_seed(42)
modell_neu = MiniGPT(vokabular_groesse, embedding_dim_neu, anzahl_heads, max_laenge)
optimizer_neu = torch.optim.Adam(modell_neu.parameters(), lr=0.003)

loss_verlauf_neu = []
for schritt in range(anzahl_schritte_neu):
    batch = hole_batch(batch_groesse)
    eingabe = batch[:, :-1]
    ziel = batch[:, 1:]

    logits = modell_neu(eingabe)
    loss = criterion(logits.reshape(-1, vokabular_groesse), ziel.reshape(-1))

    optimizer_neu.zero_grad()
    loss.backward()
    optimizer_neu.step()
    loss_verlauf_neu.append(loss.item())

print(f"Original-Training  ({anzahl_schritte} Schritte, embedding_dim={embedding_dim}): End-Loss = {sum(loss_verlauf[-50:]) / 50:.3f}")
print(f"Verändertes Training ({anzahl_schritte_neu} Schritte, embedding_dim={embedding_dim_neu}): End-Loss = {sum(loss_verlauf_neu[-50:]) / 50:.3f}")

> 💡 **Good to know:**
> Weniger Trainingsschritte oder eine kleinere `embedding_dim` führen fast immer zu einem
> **höheren** (schlechteren) End-Loss — das Modell hatte schlicht weniger Zeit oder weniger
> "Speicherplatz", um Muster zu lernen. Genau dieser Zusammenhang (mehr Kapazität + mehr Training
> = bessere Ergebnisse) ist der Kern der Vorlesungs-Erkenntnis "Skalierung, nicht neue
> Mathematik", die uns gleich am Ende noch einmal begegnet.

### Zwischenfazit

Wir haben gesehen, wie Hyperparameter den Loss beeinflussen. Jetzt kommt der spannendste Teil:
Wir lassen unser (original trainiertes) Modell selbst neue Namen erfinden.

## Schritt 7 — Inferenz: Neue Namen generieren

**Analogie — Autoregression als Kettenreaktion:** Das Modell generiert nicht den ganzen Namen auf
einmal, sondern **Zeichen für Zeichen**: Es bekommt einen Start (z. B. `<BOS>` + "A"), sagt das
wahrscheinlichste nächste Zeichen voraus, hängt es an die Sequenz an, füttert die *verlängerte*
Sequenz erneut ins Modell — und wiederholt das, bis `<EOS>` erscheint oder eine Maximallänge
erreicht ist. Genau das nennt man **Autoregression**: Jede neue Vorhersage baut auf allen vorigen
Vorhersagen auf.

Beim letzten Schritt (welches Zeichen wählen wir aus den vorhergesagten Wahrscheinlichkeiten?)
kommt **Temperature-Sampling** ins Spiel — identisch zur Vorlesungsfolie:

- **Temperature = 0:** immer das wahrscheinlichste Zeichen wählen (deterministisch, kann sich
  wiederholen).
- **Temperature ≈ 0,7:** Wahrscheinlichkeiten leicht "glätten", meiste generierte Namen wirken
  realistisch, mit gesunder Portion Abwechslung.
- **Temperature ≈ 1,5:** Wahrscheinlichkeiten stark "glätten" (fast alle Zeichen werden ähnlich
  wahrscheinlich) → chaotisches, kaum namensartiges Ergebnis.

In [ ]:
# I DO: Funktion zur autoregressiven Namensgenerierung mit Temperature-Sampling

def generiere_namen(modell, start_zeichen="", temperature=0.7, max_laenge_generierung=20):
    """Generiert einen neuen Namen Zeichen fuer Zeichen, ausgehend von 'start_zeichen'."""
    modell.eval()  # Auswertungsmodus (u.a. wichtig, falls Dropout/BatchNorm genutzt wuerden)

    # Start-Sequenz: <BOS> + optionale Startzeichen
    sequenz = [char_to_idx["<BOS>"]] + [char_to_idx[c] for c in start_zeichen]

    with torch.no_grad():  # kein Gradienten-Tracking noetig, wir trainieren hier nicht
        for _ in range(max_laenge_generierung):
            eingabe = torch.tensor([sequenz[-max_laenge:]])  # auf max_laenge begrenzen
            logits = modell(eingabe)
            letzte_logits = logits[0, -1, :]  # Vorhersage fuer die naechste Position

            if temperature == 0:
                # Deterministisch: immer das wahrscheinlichste Zeichen waehlen
                naechstes_zeichen_idx = torch.argmax(letzte_logits).item()
            else:
                # Sampling: Logits durch Temperature teilen, dann Softmax + zufaellige Auswahl
                wahrscheinlichkeiten = torch.softmax(letzte_logits / temperature, dim=0)
                naechstes_zeichen_idx = torch.multinomial(wahrscheinlichkeiten, num_samples=1).item()

            if naechstes_zeichen_idx == char_to_idx["<EOS>"]:
                break
            sequenz.append(naechstes_zeichen_idx)

    # Ersten Buchstaben (<BOS>) entfernen und Rest zu einem lesbaren Namen zusammenbauen
    generierte_zeichen = [idx_to_char[i] for i in sequenz[1:]]
    return "".join(generierte_zeichen).capitalize()

# Kurzer Test
print("Ein generierter Name:", generiere_namen(modell, start_zeichen="A", temperature=0.7))

### Zwischenfazit

Die Generierungs-Funktion steht. Jetzt bist Du dran: Vergleiche systematisch, wie sich die drei
Temperature-Stufen aus der Vorlesungsfolie auf die generierten Namen auswirken.

## Mini-Exercise — Temperature-Sampling vergleichen

> 🎯 **Your Task:**
> Vervollständige in der Zelle unten die drei `???`-Platzhalter mit den passenden
> Temperature-Werten aus der Vorlesungsfolie: **0** (deterministisch), **0,7** (balanciert) und
> **1,5** (chaotisch/kreativ). Führe die Zelle danach mehrfach aus und beobachte: Bleibt der
> Name bei Temperature 0 immer gleich? Wie "namensartig" wirken die Ergebnisse bei Temperature
> 1,5 noch?

In [ ]:
# =========================================================================
# 🎯 EXERCISE: Temperature-Werte einsetzen und Effekt vergleichen
# =========================================================================
# Instruction: Ersetze die drei ??? durch 0, 0.7 bzw. 1.5 (siehe Anleitung oben).

temperature_deterministisch = ???
temperature_balanciert = ???
temperature_chaotisch = ???

for temperature, bezeichnung in [
    (temperature_deterministisch, "Deterministisch (Temperature 0)"),
    (temperature_balanciert, "Balanciert (Temperature 0.7)"),
    (temperature_chaotisch, "Chaotisch (Temperature 1.5)"),
]:
    generierte_namen = [generiere_namen(modell, start_zeichen="A", temperature=temperature) for _ in range(5)]
    print(f"{bezeichnung}: {generierte_namen}")

> ⚠️ **Common Pitfall:**
> Bei Temperature 0 wählt das Modell **immer** dieselbe Fortsetzung für denselben Start — deshalb
> siehst Du oben fünfmal denselben Namen. Das ist kein Bug, sondern exakt das erwartete
> Verhalten: "wähle immer das Wahrscheinlichste" lässt keinen Platz für Zufall.

> 💡 **Good to know:**
> Real-World-Chatbots wie ChatGPT nutzen für kreatives Schreiben typischerweise
> Temperature-Werte um 0,7–0,9 — genug Abwechslung für natürlich wirkende Antworten, aber nicht
> so viel, dass der Text in Kauderwelsch abdriftet. Für Aufgaben, bei denen es eine "richtige"
> Antwort gibt (z. B. eine Übersetzung oder ein Rechenergebnis), wird dagegen oft
> Temperature = 0 verwendet.

## Self-Check — Reflexionsfrage: Vom Mini-Modell zum Produktions-LLM

> 🎯 **Your Task:**
> Beantworte die folgende Frage als `print()`-Ausgabe in der Zelle unten:
>
> Unser `MiniGPT` kann gerade einmal plausible **Vornamen** erzeugen — kein einziges vollständiges
> Wort, geschweige denn einen Satz. Was müsste sich ändern (nenne **mindestens zwei** konkrete
> Stellschrauben aus diesem Notebook), damit ein Modell wie dieses ganze Sätze plausibel
> vervollständigen könnte? Verbinde Deine Antwort mit der Vorlesungs-Erkenntnis: Ist dafür **neue
> Mathematik** nötig, oder etwas anderes?

In [ ]:
# TODO: Trage Deine Antwort als Kommentar bzw. print()-Ausgabe ein.

antwort = "???"

print(f"Antwort: {antwort}")

## Summary & Key Takeaways

- Ein **Character-Level-Tokenizer** übersetzt Text in Zahlen (und zurück) über zwei einfache
  Dictionaries (`char_to_idx`, `idx_to_char`) plus drei Spezial-Tokens (`<BOS>`, `<EOS>`,
  `<PAD>`).
- **`MiniGPT`**-Architektur: Token-Embedding + Positions-Embedding → ein Transformer-Block
  (Self-Attention mit **causal masking** + MLP, je mit Residual Connection und Normalisierung) →
  Output-Projektion auf die Vokabulargröße. Exakt dieselben vier Bausteine wie auf der
  Vorlesungsfolie "Architektur-Übersicht (microgpt)".
- Der **Trainingsloop** folgt demselben Fünf-Schritte-Muster wie in den vorigen Kapiteln
  (Forward Pass → Loss → `zero_grad()` → `backward()` → `step()`) — der einzige Unterschied: Wir
  sagen das nächste Zeichen an **jeder** Position gleichzeitig voraus, nicht nur eine einzige
  Ausgabe pro Beispiel.
- Der Loss unseres Mini-Modells sinkt deutlich unter den Referenzwert für zufälliges Raten
  (ln(Vokabulargröße)) — dasselbe Muster wie microgpts Sprung von ~3,3 auf ~2,37 auf der
  Vorlesungsfolie.
- **Autoregression** (Zeichen für Zeichen generieren, jede Vorhersage baut auf der vorigen auf)
  plus **Temperature-Sampling** (0 = deterministisch, ~0,7 = balanciert, ~1,5 = chaotisch)
  steuern, wie ein Sprachmodell neue Sequenzen erzeugt — vom winzigen Namensgenerator bis zu
  ChatGPT.
- **Kernbotschaft:** Ein LLM ist algorithmisch **kein** neues, unverständliches Konzept — es
  besteht aus denselben Bausteinen (Embeddings, lineare Schichten, Aktivierungsfunktionen,
  Backpropagation), die Du schon aus den vorigen Kapiteln kennst. Der Unterschied zwischen
  unserem `MiniGPT` und einem Produktions-LLM wie GPT-4 liegt fast ausschließlich in
  **Skalierung**: mehr Parameter, mehr und diversere Trainingsdaten, mehr Trainingsschritte und
  zusätzliches Post-Training (SFT, RLHF) — nicht in neuer Mathematik.